In [ ]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, PayloadSchemaType
)
from ingestion.chunker import Chunk
import config
import logging
from tqdm import tqdm
import uuid

logger = logging.getLogger(__name__)

In [ ]:
def get_qdrant_client() -> QdrantClient:
    return QdrantClient(
        url=config.QDRANT_URL,
        api_key=config.QDRANT_API_KEY,
    )


def setup_collection(client: QdrantClient, recreate: bool = False):
    """Create the Qdrant collection if it doesn't exist."""
    existing = [c.name for c in client.get_collections().collections]

    if config.COLLECTION_NAME in existing:
        if recreate:
            client.delete_collection(config.COLLECTION_NAME)
            logger.info(f"Deleted existing collection: {config.COLLECTION_NAME}")
        else:
            logger.info(f"Collection already exists: {config.COLLECTION_NAME}")
            return

    client.create_collection(
        collection_name=config.COLLECTION_NAME,
        vectors_config=VectorParams(
            size=config.EMBED_DIMENSION,
            distance=Distance.COSINE,
        ),
    )

    # Index payload fields for fast filtering
    for field_name in ("domain", "section_id", "parent_section"):
        client.create_payload_index(
            collection_name=config.COLLECTION_NAME,
            field_name=field_name,
            field_schema=PayloadSchemaType.KEYWORD,
        )

    logger.info(f"Created collection: {config.COLLECTION_NAME}")

In [ ]:
recreate = False # Set to True to drop and recreate the collection

client = get_qdrant_client()
setup_collection(client, recreate=recreate)

### Load Chunks from File

In [18]:
import json

raw_data = json.load(open("./data/debug/ch591_noise_chunks.json", "r"))
chunks = [Chunk(**item) for item in raw_data]

In [ ]:
model = SentenceTransformer(config.EMBED_MODEL)

In [ ]:
texts = [chunk.text for chunk in chunks]

In [ ]:
prefixed = [f"Represent this sentence for searching relevant passages: {t}" for t in texts]

In [ ]:
all_embeddings = []
for i in tqdm(range(0, len(prefixed), config.EMBED_BATCH_SIZE)):
    batch = prefixed[i:i + config.EMBED_BATCH_SIZE]
    embeddings = model.encode(batch, normalize_embeddings=True)
    all_embeddings.extend(embeddings)

In [ ]:
 # Build Qdrant points
points = []
for chunk, embedding in zip(chunks, all_embeddings):
    points.append(PointStruct(
        id=str(uuid.uuid4()),
        vector=embedding.tolist(),
        payload={
            "chunk_id": chunk.chunk_id,
            "domain": chunk.domain,
            "section_id": chunk.section_id,
            "section_title": chunk.section_title,
            "parent_section": chunk.parent_section,
            "text": chunk.text,
            "source_file": chunk.source_file,
            "page": chunk.page,
        }
    ))

In [ ]:
# Upsert in batches of 100
for i in tqdm(range(0, len(points), 100)):
    client.upsert(
        collection_name=config.COLLECTION_NAME,
        points=points[i:i + 100],
    )